In [ ]:
import xgboost as xgb
import pandas as pd


try:
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model('triage_xgboost.json')
except Exception as e:
    print(f"Error loading model: {e}")

TRIAGE_MAP = {0: "Green", 1: "Yellow", 2: "Red"}
RED_FLAG_KEYWORDS = ["dizzy", "chest pain", "fever", "fainting", "blood", "shortness of breath"]

def analyze_patient(sensor_packet: dict) -> dict:
    """
    Consumes Shashwat's JSON packet and returns the locked triage contract.
    """
   
    urine_rgb = sensor_packet.get('urine_rgb', [255.0, 234.0, 112.0])
    if len(urine_rgb) != 3:
        urine_rgb = [255.0, 234.0, 112.0]

    
    features = pd.DataFrame([{
        'ecg_hr': sensor_packet.get('ecg_hr', 75.0),
        'bp_sys': sensor_packet.get('bp_sys', 120.0),
        'bp_dia': sensor_packet.get('bp_dia', 80.0),
        'spo2': sensor_packet.get('spo2', 98.0),
        'temperature': sensor_packet.get('temperature', 37.0),
        'urine_r': urine_rgb[0],
        'urine_g': urine_rgb[1],
        'urine_b': urine_rgb[2]
    }])
    
    # XGBoost Inference
    probs = xgb_model.predict_proba(features)[0]
    predicted_class_idx = int(probs.argmax())
    confidence = float(probs[predicted_class_idx])
    ai_triage = TRIAGE_MAP[predicted_class_idx]
    
    # 4. Symptom Extraction
    speech_text = sensor_packet.get('patient_speech_text', "").lower()
    symptoms = [kw for kw in RED_FLAG_KEYWORDS if kw in speech_text]
    
    # Escalate Red if severe 
    if symptoms and ai_triage != "Red":
        ai_triage = "Red"
        confidence = 0.99 
        
    return {
        "triage": ai_triage,
        "confidence": round(confidence, 2)
    }